# PyOD with `cuml.accel` (GPU-backed scikit-learn)

Many PyOD detectors wrap **scikit-learn** estimators (for example `IsolationForest`, `PCA`, `NearestNeighbors`, `KMeans`). The RAPIDS library provides **`cuml.accel`**, which swaps supported scikit-learn estimators for **cuML** implementations on the GPU where available, with automatic CPU fallback otherwise.

**How it applies to PyOD**

- PyOD imports scikit-learn inside each model module (for example `pyod.models.iforest` imports `sklearn.ensemble.IsolationForest`).
- To use GPU-backed estimators in the **same Jupyter session** after you have already imported PyOD on CPU, follow the RAPIDS pattern: load the extension, **re-import** the relevant `sklearn` symbols, then **`importlib.reload`** the PyOD model modules so their `from sklearn...` lines bind to the accelerated classes.
- Alternatively, enable acceleration **before** any `sklearn` / PyOD import: `cuml.accel.install()`, the `CUML_ACCEL_ENABLED=1` environment variable, or `python -m cuml.accel your_script.py`.

For each of five detectors (IForest, PCA, KNN, CBLOF, HDBSCAN), we time **`fit`** and **`predict`** on held-out test data **separately** with normal PyOD (CPU sklearn), **then** (after `%load_ext cuml.accel` and reload) repeat the same workflow.

**Requirements**

- NVIDIA GPU, CUDA, and a RAPIDS/cuML environment ([RAPIDS install](https://docs.rapids.ai/install)).
- Compatible `pyod` and scikit-learn versions ([cuml.accel limitations](https://docs.rapids.ai/api/cuml/stable/cuml-accel/limitations/)).

**Note:** Speedups depend on dataset size, GPU, and whether each estimator is fully accelerated or falls back to CPU. Small arrays can look slower on GPU because of transfer overhead. This demo uses 20K samples to keep runtime short; for meaningful GPU speedups, increase `N_TRAIN` to 100K+ (some algorithms like HDBSCAN may require significant CPU time at that scale).

## Setup: synthetic data and timing helper

In [1]:
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from pyod.utils.data import generate_data

RANDOM_STATE = 42
# N_TRAIN = 1_000_000
# N_TEST = 200_000
N_TRAIN = 20_000
N_TEST = 5_000
N_FEATURES = 32
CONTAMINATION = 0.1

X_train, X_test, y_train, y_test = generate_data(
    n_train=N_TRAIN,
    n_test=N_TEST,
    n_features=N_FEATURES,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    behaviour="new",
)
X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

print("X_train:", X_train.shape, X_train.dtype, " X_test:", X_test.shape)

X_train: (20000, 32) float32  X_test: (5000, 32)


In [2]:
def time_fit_and_predict(detector, X_tr, X_te):
    """Wall times (seconds) for fit on train, then predict on test."""
    t0 = time.perf_counter()
    detector.fit(X_tr)
    t_fit = time.perf_counter() - t0
    t0 = time.perf_counter()
    detector.predict(X_te)
    t_predict = time.perf_counter() - t0
    return t_fit, t_predict


cpu_times = {}
gpu_times = {}

---
## 1. Isolation Forest (`IForest`)

Wraps `sklearn.ensemble.IsolationForest`.

In [3]:
from pyod.models.iforest import IForest

clf = IForest(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["IForest"] = {"fit": tf, "predict": tp}
print(f"IForest CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

IForest CPU (sklearn): fit=0.6424s  predict=0.0403s


---
## 2. PCA reconstruction-based detector (`PCA`)

Uses `sklearn.decomposition.PCA` internally.

In [4]:
from pyod.models.pca import PCA

clf = PCA(n_components=16, random_state=RANDOM_STATE)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["PCA"] = {"fit": tf, "predict": tp}
print(f"PCA CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

PCA CPU (sklearn): fit=0.0182s  predict=0.0034s


---
## 3. kNN distances (`KNN`)

Uses `sklearn.neighbors.NearestNeighbors` (and related neighbor search).

In [5]:
from pyod.models.knn import KNN

clf = KNN(n_neighbors=20, method="mean", metric="euclidean", n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["KNN"] = {"fit": tf, "predict": tp}
print(f"KNN CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

KNN CPU (sklearn): fit=0.9340s  predict=0.4733s


---
## 4. CBLOF with default `KMeans` (`CBLOF`)

Uses `sklearn.cluster.KMeans` for clustering by default.

In [6]:
from pyod.models.cblof import CBLOF

clf = CBLOF(n_clusters=16, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["CBLOF"] = {"fit": tf, "predict": tp}
print(f"CBLOF CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

CBLOF CPU (sklearn): fit=1.2596s  predict=0.0028s


---
## 5. Kernel density (`KDE`)

Wraps `sklearn.neighbors.KernelDensity`. Using `metric='euclidean'` aligns with common `cuml.accel` support for density estimation (see RAPIDS limitations for your metric).

---
## 6. HDBSCAN (`HDBSCAN`)

Uses `sklearn.cluster.HDBSCAN` for clustering and `sklearn.neighbors.NearestNeighbors` for scoring new points. Requires a recent scikit-learn that includes `sklearn.cluster.HDBSCAN`.

In [7]:
from pyod.models.hdbscan import HDBSCAN

clf = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    n_jobs=-1,
    contamination=CONTAMINATION,
)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
cpu_times["HDBSCAN"] = {"fit": tf, "predict": tp}
print(f"HDBSCAN CPU (sklearn): fit={tf:.4f}s  predict={tp:.4f}s")

HDBSCAN CPU (sklearn): fit=18.9151s  predict=1.1410s


---
## Enable `cuml.accel` and reload PyOD model modules

Run the next two cells **once**, then run the GPU timing cells below. This matches the RAPIDS tutorial: after `%load_ext`, re-import sklearn estimators and reload PyOD modules that wrap them.

In [8]:
# %load_ext cuml.accel
import cuml
cuml.accel.install(log_level="debug")


[cuml.accel] Enabled managed memory.
[cuml.accel] Accelerator installed.


---
## Same six workflows with `cuml.accel` enabled

Hyperparameters match the CPU section. Timings are **`fit`** on `X_train` and **`predict`** on `X_test` (binary labels; PyOD may call `decision_function` internally).

In [9]:
clf = IForest(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["IForest"] = {"fit": tf, "predict": tp}
print(f"IForest GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

IForest GPU (cuml.accel): fit=0.6482s  predict=0.0403s


In [10]:
clf = PCA(n_components=16, random_state=RANDOM_STATE)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["PCA"] = {"fit": tf, "predict": tp}
print(f"PCA GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

PCA GPU (cuml.accel): fit=0.0182s  predict=0.0022s


In [11]:
clf = KNN(n_neighbors=20, method="mean", metric="euclidean", n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["KNN"] = {"fit": tf, "predict": tp}
print(f"KNN GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

KNN GPU (cuml.accel): fit=2.0838s  predict=0.4666s


In [12]:
clf = CBLOF(n_clusters=16, random_state=RANDOM_STATE, n_jobs=-1)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["CBLOF"] = {"fit": tf, "predict": tp}
print(f"CBLOF GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

CBLOF GPU (cuml.accel): fit=0.5964s  predict=0.0028s


In [13]:
clf = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    n_jobs=-1,
    contamination=CONTAMINATION,
)
tf, tp = time_fit_and_predict(clf, X_train, X_test)
gpu_times["HDBSCAN"] = {"fit": tf, "predict": tp}
print(f"HDBSCAN GPU (cuml.accel): fit={tf:.4f}s  predict={tp:.4f}s")

HDBSCAN GPU (cuml.accel): fit=18.6911s  predict=0.9222s


## Summary table

In [14]:
rows = []
for name in ("IForest", "PCA", "KNN", "CBLOF", "HDBSCAN"):
    cf = cpu_times[name]["fit"]
    cp = cpu_times[name]["predict"]
    gf = gpu_times[name]["fit"]
    gp = gpu_times[name]["predict"]
    sf = cf / gf if gf > 0 else float("nan")
    sp = cp / gp if gp > 0 else float("nan")
    rows.append((name, cf, cp, gf, gp, sf, sp))

try:
    import pandas as pd

    display(
        pd.DataFrame(
            rows,
            columns=[
                "detector",
                "cpu_fit_s",
                "cpu_predict_s",
                "gpu_fit_s",
                "gpu_predict_s",
                "fit_speedup_cpu/gpu",
                "predict_speedup_cpu/gpu",
            ],
        )
    )
except Exception:
    hdr = (
        f"{'detector':<10} {'cpu_fit':>10} {'cpu_pred':>10} "
        f"{'gpu_fit':>10} {'gpu_pred':>10} {'fit_sp':>8} {'pred_sp':>8}"
    )
    print(hdr)
    for name, cf, cp, gf, gp, sf, sp in rows:
        print(
            f"{name:<10} {cf:10.4f} {cp:10.4f} {gf:10.4f} {gp:10.4f} "
            f"{sf:8.2f}x {sp:8.2f}x"
        )

,detector,cpu_fit_s,cpu_predict_s,gpu_fit_s,gpu_predict_s,fit_speedup_cpu/gpu,predict_speedup_cpu/gpu
0,IForest,0.642424,0.040312,0.648229,0.040286,0.991045,1.000663
1,PCA,0.018247,0.003361,0.018193,0.002169,1.002958,1.549451
2,KNN,0.933982,0.473257,2.083787,0.466610,0.448214,1.014245
3,CBLOF,1.259573,0.002791,0.596367,0.002752,2.112078,1.014267
4,HDBSCAN,18.915053,1.141021,18.691098,0.922229,1.011982,1.237243


## Optional: other ways to enable acceleration

- **Programmatic (before sklearn / pyod):** `cuml.accel.install()`
- **CLI:** `python -m cuml.accel your_script.py`
- **Environment:** `export CUML_ACCEL_ENABLED=1`

See [logging and profiling](https://docs.rapids.ai/api/cuml/stable/cuml-accel/logging-and-profiling/) to see GPU vs CPU fallback per call.